# Two Agents — Code Writer & Code Executor

## Setup

In [ ]:
# autogen-agentchat 0.7.x is compatible with Python 3.14
# ConversableAgent and LocalCommandLineCodeExecutor are in autogen-agentchat + autogen-ext
%pip install -q autogen-agentchat autogen-ext[openai] python-dotenv openai

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path
import pprint

load_dotenv("untitled.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found — check untitled.env")

In [ ]:
import autogen_agentchat
print("autogen-agentchat version:", autogen_agentchat.__version__)

## Create Agents

### Agent 1 — Code Executor

In [ ]:
from autogen_agentchat.agents import CodeExecutorAgent
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

work_dir = Path("coding")
work_dir.mkdir(exist_ok=True)

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

code_executor_agent = CodeExecutorAgent(
    name="code_executor_agent",
    code_executor=executor,
)

### Agent 2 — Code Writer

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

code_writer_system_message = """
You have been given coding capability to solve tasks using Python code.
In the following cases, suggest python code (in a python coding block) or shell script
(in a sh coding block) for the user to execute.
    1. When you need to collect info, use the code to output the info you need, for
       example, browse or search the web, download/read a file, print the content of
       a webpage or a file, get the current date/time, check the operating system.
       After sufficient info is printed and the task is ready to be solved based on
       your language skill, you can solve the task by yourself.
    2. When you need to perform some task with code, use the code to perform the task
       and output the result. Finish the task smartly.
Solve the task step by step if you need to. Be clear which step uses code and which
step uses your language skill. Always indicate the script type in the code block.
Do not suggest incomplete code. Don't use a code block if it's not intended to be
executed. Use 'print' for outputs. Check the execution result returned by the user.
When the task is done, reply with TERMINATE.
"""

openai_client = OpenAIChatCompletionClient(
    model="gpt-4",
    api_key=OPENAI_API_KEY,
)

code_writer_agent = AssistantAgent(
    name="code_writer",
    model_client=openai_client,
    system_message=code_writer_system_message,
)

## Run the Two-Agent Chat

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(10)

team = RoundRobinGroupChat(
    [code_writer_agent, code_executor_agent],
    termination_condition=termination,
)

task = (
    "Write Python code for permutation for the word ALGEBRA. "
    "Use an optimised way to calculate it. I want the final result count."
)

result = await Console(team.run_stream(task=task))

## Inspect Results

In [ ]:
# All messages in the conversation
for msg in result.messages:
    print(f"[{msg.source}]: {msg.content[:200]}")
    print()

In [ ]:
# Final summary (last message)
print("=== FINAL RESULT ===")
print(result.messages[-1].content)

In [ ]:
# Stop reason
print("Stop reason:", result.stop_reason)